# 06 Error Analysis

Analyze Common Voice Yue predictions by accent, age, gender, sentence length, and duration.

In [ ]:
from pathlib import Path
import os
import sys

def running_in_colab():
    return 'google.colab' in sys.modules or Path('/content').exists()

def mount_drive_if_available():
    if not running_in_colab():
        return
    try:
        from google.colab import drive
        if not Path('/content/drive').exists():
            drive.mount('/content/drive')
    except Exception as exc:
        print(f'Google Drive mount skipped: {exc}')

def find_project_root():
    mount_drive_if_available()
    cwd = Path.cwd().resolve()
    env_root = os.getenv('PROJECT_ROOT')
    candidates = []
    if env_root:
        candidates.append(Path(env_root))
    candidates += [cwd, *cwd.parents]
    candidates += [
        Path('/content/Cantonese-Speech-AI'),
        Path('/content/drive/MyDrive/Cantonese-Speech-AI'),
        Path('/content/drive/My Drive/Cantonese-Speech-AI'),
        Path(r'D:/GitHub/Cantonese-Speech-AI'),
        Path(r'D:\\GitHub\\Cantonese-Speech-AI'),
        Path('/mnt/d/GitHub/Cantonese-Speech-AI'),
    ]
    for candidate in candidates:
        if (candidate / 'src' / 'cantonese_speech_ai').exists():
            return candidate.resolve()
    checked = '\n'.join(str(path) for path in candidates[:12])
    raise FileNotFoundError(
        'Cannot find the Cantonese-Speech-AI project folder in this runtime.\n\n'
        'Colab fix:\n'
        '1. Upload or copy the whole Cantonese-Speech-AI folder to Google Drive.\n'
        '2. Expected path: /content/drive/MyDrive/Cantonese-Speech-AI\n'
        '3. The folder must contain src/cantonese_speech_ai and Mozilla-HK-Speech-datasets.\n'
        '4. Then restart runtime and run this cell again.\n\n'
        'Alternative: set os.environ[\'PROJECT_ROOT\'] to your folder path before this cell.\n\n'
        f'Current cwd: {cwd}\nChecked:\n{checked}'
    )

ROOT = find_project_root()
os.environ['PROJECT_ROOT'] = str(ROOT)
default_dataset = ROOT / 'Mozilla-HK-Speech-datasets' / 'cv-corpus-26.0-2026-06-12' / 'yue'
os.environ.setdefault('COMMON_VOICE_YUE_ROOT', str(default_dataset))

SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd

from cantonese_speech_ai.common_voice import CommonVoicePaths, prepare_split
from cantonese_speech_ai.metrics import char_error_rate, word_error_rate

paths = CommonVoicePaths.from_env()
if not paths.split_path('train').exists():
    raise FileNotFoundError(f'Dataset train.tsv not found: {paths.split_path("train")}')
ROOT, paths
test = pd.DataFrame(prepare_split('test', paths))


## Load or create prediction table

Expected prediction CSV columns: `path`, `prediction`. If no prediction file exists yet, this cell creates an oracle table so the analysis code can be tested.

In [ ]:
prediction_path = ROOT / 'data' / 'predictions' / 'common_voice_yue_test_predictions.csv'

if prediction_path.exists():
    predictions = pd.read_csv(prediction_path, encoding='utf-8')
else:
    predictions = test[['path', 'normalized_sentence']].rename(columns={'normalized_sentence': 'prediction'})

analysis = test.merge(predictions[['path', 'prediction']], on='path', how='left')
analysis['prediction'] = analysis['prediction'].fillna('')
analysis[['path', 'sentence', 'prediction']].head()

## Compute errors

In [ ]:
analysis['sentence_length'] = analysis['sentence'].fillna('').str.len()
analysis['cer'] = analysis.apply(lambda row: char_error_rate(row['normalized_sentence'], row['prediction']), axis=1)
analysis['wer'] = analysis.apply(lambda row: word_error_rate(row['normalized_sentence'], row['prediction']), axis=1)
analysis[['cer', 'wer', 'duration_sec', 'sentence_length']].describe()

## Metadata breakdowns

In [ ]:
for column in ['accents', 'age', 'gender']:
    analysis[column] = analysis[column].fillna('unknown').replace('', 'unknown')

breakdowns = {
    'accent_cer': analysis.groupby('accents')['cer'].agg(['count', 'mean']).sort_values('count', ascending=False).head(10),
    'age_cer': analysis.groupby('age')['cer'].agg(['count', 'mean']).sort_values('count', ascending=False),
    'gender_cer': analysis.groupby('gender')['cer'].agg(['count', 'mean']).sort_values('count', ascending=False),
}
breakdowns